In [21]:
import yfinance as yf
import pandas as pd
import requests
import os

def isin_to_ticker(isin):
    """Convert ISIN to Yahoo ticker using Yahoo Finance search API."""
    url = f"https://query2.finance.yahoo.com/v1/finance/search?q={isin}"
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    try:
        response = requests.get(url, headers=headers)
        data = response.json()
        if data.get('quotes') and len(data['quotes']) > 0:
            return data['quotes'][0]['symbol']  # First/best match
    except Exception as e:
        print(f"Error fetching ticker for {isin}: {e}")
    return None

def download_close_prices(
    symbols,  # List of ISINs or tickers
    start_date,
    end_date,
    output_csv="close_prices.csv"
):
    print("Downloading data...")
    
    # Map ISINs to tickers
    tickers = []
    for symbol in symbols:
        ticker = symbol
        if len(symbol) == 12 and symbol[0].isalpha() and symbol[11].isdigit():  # Basic ISIN check
            print(f"Converting ISIN {symbol}...")
            ticker = isin_to_ticker(symbol)
        if ticker:
            tickers.append(ticker)
            print(f"✓ {symbol} → {ticker}")
        else:
            print(f"✗ No ticker found for {symbol}")
    
    if not tickers:
        raise ValueError("No valid tickers resolved.")
    
    print(f"Downloading {len(tickers)} tickers...")
    
    df = yf.download(
        tickers,
        start=start_date,
        end=end_date,
        interval="1mo",
        progress=False,
        auto_adjust=True
    )
    
    if df.empty:
        raise ValueError("No data downloaded.")
    
    # Keep only Close prices
    close_df = df["Close"]
    
    # If only one ticker, make it a DataFrame
    if isinstance(close_df, pd.Series):
        close_df = close_df.to_frame(name=tickers[0])
    
    close_df = close_df.ffill()
    close_df.index.name = "Date"
    close_df.to_csv(output_csv, sep=";", decimal=",")
    
    print(f"✅ Data saved to {output_csv}")
    print(f"Shape: {close_df.shape}")

if __name__ == "__main__":
    SYMBOLS = [
        # Examples - replace with your ISINs or tickers
        "DE000A2T0VU",
        "LU3061478973",
        "IE00BFMXXD54",
        "IE00B3XXRP09",
        "LU1829221024",
        "LU1681042609"
    ]
    
    START_DATE = "2015-01-01"
    END_DATE = "2026-02-09"  # Up to current date
    
    download_close_prices(
        symbols=SYMBOLS,
        start_date=START_DATE,
        end_date=END_DATE,
        output_csv="close_prices_new.csv"
    )


✓ DE000A2T0VU → DE000A2T0VU
Converting ISIN LU3061478973...
✓ LU3061478973 → XDEF.DE
Converting ISIN IE00BFMXXD54...
✓ IE00BFMXXD54 → VUAA.L
Converting ISIN IE00B3XXRP09...
✓ IE00B3XXRP09 → VUSA.L
Converting ISIN LU1829221024...
✓ LU1829221024 → NASD.L
Converting ISIN LU1681042609...
✓ LU1681042609 → CEUG.DE


/Users/ascaravelli/Documents/GitHub/Congress_stock_analysis/venv/lib/python3.14/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
/Users/ascaravelli/Documents/GitHub/Congress_stock_analysis/venv/lib/python3.14/site-packages/yfinance/scrapers/history.py:201: Pandas4Warning: Timestamp.utcnow is deprecated and will be removed in a future version. Use Timestamp.now('UTC') instead.
  dt_now = pd.Timestamp.utcnow()
$DE000A2T0VU: possibly delisted; no timezone found

1 Failed download:
['DE000A2T0VU']: possibly delisted; no timezone found


✅ Data saved to close_prices_new.csv
Shape: (134, 6)
